In [ ]:
# 1. find all audio history files
# 2. load and combine them into one dataframe
# 3. check the shape and look at a few rows

In [15]:
import pandas as pd
import glob
import numpy as np

In [5]:
files = glob.glob("../data/raw/spotify_history/Streaming_History_Audio_*.json")
print(f"Found {len(files)} files")

Found 6 files


In [8]:
history = pd.concat([pd.read_json(f) for f in files], ignore_index=True)
print("Total plays:", history.shape)


Total plays: (8345, 23)


In [9]:
# 3. look at a few rows
history[["master_metadata_track_name", "master_metadata_album_artist_name",
         "ms_played", "skipped"]].head()

,master_metadata_track_name,master_metadata_album_artist_name,ms_played,skipped
0,Khwab,Iqlipse Nova,141160,False
1,Khwab,Iqlipse Nova,17754,False
2,Teri Yaad,Aditya Rikhari,230649,False
3,Khwab,Iqlipse Nova,158911,False
4,Main Kho Gaya,Kushagra,77866,False


In [10]:
history = history.dropna(subset=["master_metadata_track_name"])
print("after dropping non-music rows:", history.shape)

after dropping non-music rows: (8339, 23)


In [11]:
history = history[history["ms_played"] > 5000]
print("after removing accidental plays:", history.shape)

after removing accidental plays: (7183, 23)


In [14]:
history = history[["master_metadata_track_name", "master_metadata_album_artist_name",
                    "ms_played", "skipped"]].copy()

# 4. build a readable track identifier
history["track"] = history["master_metadata_track_name"] + " - " + history["master_metadata_album_artist_name"]

history.head()

,master_metadata_track_name,master_metadata_album_artist_name,ms_played,skipped,track
0,Khwab,Iqlipse Nova,141160,False,Khwab - Iqlipse Nova
1,Khwab,Iqlipse Nova,17754,False,Khwab - Iqlipse Nova
2,Teri Yaad,Aditya Rikhari,230649,False,Teri Yaad - Aditya Rikhari
3,Khwab,Iqlipse Nova,158911,False,Khwab - Iqlipse Nova
4,Main Kho Gaya,Kushagra,77866,False,Main Kho Gaya - Kushagra


In [17]:
# 1. estimate each song's length (longest ms_played for that track = full length)
history["length"] = history.groupby("track")["ms_played"].transform("max")
history["completion"] = history["ms_played"] / history["length"] 

In [18]:
tracks = history.groupby("track").agg(
    plays=("ms_played", "count"),
    avg_completion=("completion", "mean"),
    skip_rate=("skipped", "mean")
).reset_index()

In [20]:
plays_norm = np.log1p(tracks["plays"]) / np.log1p(tracks["plays"]).max()
tracks["score"] = (0.4 * plays_norm
                    + 0.4 * tracks["avg_completion"]
                    + 0.2 * (1 - tracks["skip_rate"]))

tracks = tracks.sort_values("score", ascending=False)
tracks.head(50)

,track,plays,avg_completion,skip_rate,score
971,Oonchi Oonchi Deewarein - Manan Bhardwaj,56,0.958908,0.035714,0.940442
443,Haan Ke Haan - Sohail Sen,63,0.868546,0.031746,0.915520
1096,Ratiyaan - Hansika Pareek,76,0.829004,0.039474,0.914807
203,Chaand Baaliyan - Aditya A,62,0.896269,0.112903,0.908959
17,Aaj Se Teri - Amit Trivedi,35,0.982300,0.057143,0.904138
528,Ishq Hai - Anurag Saikia,84,0.798295,0.142857,0.890747
1161,Samjho Na - Aditya Rikhari,47,0.897469,0.085106,0.890515
1342,Thande Pahadon Mein - I-Popstar: Vol. 1 - Jaya...,24,1.000000,0.000000,0.889816
1381,"Tu Hain Toh (From ""Mr. And Mrs. Mahi"") - Bunny",64,0.841207,0.140625,0.884204
954,O Saathi - Atif Aslam,47,0.851421,0.063830,0.876351


In [21]:
tracks.to_csv("../data/processed/my_taste_scores.csv", index=False)
print(tracks.shape)

(1553, 5)
